In [ ]:
import pandas as pd

# =========================
# 1. Read files
# =========================
medals = pd.read_csv("data/olympic_medals.csv")
gdp = pd.read_csv("data/GDP.csv")

# =========================
# 2. Keep only needed columns
# =========================
medals = medals[
    [
        "discipline_title",
        "event_title",
        "event_gender",
        "medal_type",
        "participant_type",
        "country_name",
        "Area",
        "Year",
        "game_season",
        "Connect",
    ]
].copy()

gdp = gdp[
    [
        "Country Name",
        "Country Code",
        "Year",
        "Value",
        "AREA",
        "Connect",
        "Olympic-Year",
    ]
].copy()

# Rename GDP columns to match output naming
gdp = gdp.rename(
    columns={
        "Country Name": "Country",
        "Value": "GDP",
        "AREA": "Area_gdp",
    }
)

# =========================
# 3. Clean data types
# =========================
medals["Year"] = pd.to_numeric(medals["Year"], errors="coerce")
gdp["Year"] = pd.to_numeric(gdp["Year"], errors="coerce")

medals = medals.dropna(subset=["Year", "country_name", "medal_type", "Connect"])
gdp = gdp.dropna(subset=["Year", "GDP", "Connect"])

medals["Year"] = medals["Year"].astype(int)
gdp["Year"] = gdp["Year"].astype(int)

# Optional: keep only Olympics from 1960 onwards
medals = medals[medals["Year"] >= 1960]
gdp = gdp[gdp["Year"] >= 1960]

# Optional: if you want Summer only, uncomment this
# medals = medals[medals["game_season"] == "Summer"]

# =========================
# 4. Deduplicate medal records
# =========================
# We count one medal-winning result per country per event.
# This avoids accidental double counting if the same result appears multiple times.
medals_unique = medals.drop_duplicates(
    subset=[
        "Connect",
        "discipline_title",
        "event_title",
        "event_gender",
        "medal_type",
        "country_name",
        "participant_type",
    ]
).copy()

# =========================
# 5. Aggregate medal counts
# =========================
# Medal count by Year × Country × MedalType
medal_count = (
    medals_unique.groupby(
        ["Year", "Connect", "country_name", "Area", "medal_type"],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "country_name": "Country",
            "medal_type": "MedalType",
            "size": "MedalCount",
        }
    )
)

# =========================
# 6. Merge with GDP using Connect + Year
# =========================
merged = pd.merge(
    medal_count,
    gdp[["Year", "Connect", "Country", "GDP", "Area_gdp"]],
    on=["Year", "Connect"],
    how="inner",
    suffixes=("", "_gdp")
)

# =========================
# 7. Resolve country/area columns
# =========================
# Keep medal file's country/area as primary, but fall back to GDP if needed
merged["Country"] = merged["Country"].fillna(merged["Country_gdp"]) if "Country_gdp" in merged.columns else merged["Country"]
merged["Area"] = merged["Area"].fillna(merged["Area_gdp"]) if "Area_gdp" in merged.columns else merged["Area"]

# Keep only the columns needed for D3
final_df = merged[["Year", "Country", "Area", "MedalType", "GDP", "MedalCount"]].copy()

# Remove invalid GDP values
final_df = final_df.dropna(subset=["GDP"])
final_df = final_df[final_df["GDP"] > 0]

# Sort for readability
final_df = final_df.sort_values(["Year", "Country", "MedalType"]).reset_index(drop=True)

# =========================
# 8. Save output
# =========================
final_df.to_csv("gdp_medals.csv", index=False)

print("Created gdp_medals.csv")
print(final_df.head(10))
print(f"Rows: {len(final_df)}")